# TRABAJO FINAL UT7 - AGRUPACIÓN

En este trabajo vamos a aplicar distintas técnicas de aprendizaje no supervisado (clustering) sobre un dataset de clientes de un centro comercial. La idea es segmentar a los clientes en grupos según sus características, para que el centro comercial pueda dirigir mejor sus campañas de marketing.

También haremos Market Basket Analysis con el algoritmo Apriori y detección de anomalías con Bosques de Aislamiento.

---
## 1. EDA Y PREPROCESADO DE LOS DATOS (2 puntos)

### 1.1 Importar librerías y dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
# Cargamos el dataset de clientes del centro comercial
# Este dataset tiene info de 200 clientes: género, edad, ingresos y puntaje de gasto
df = pd.read_excel('Cliente_Centro_Comercial.xlsx')
df.head(10)

In [ ]:
# Vemos la forma del dataset
print('Dimensiones del dataset:', df.shape)
print()
print('Tipos de datos:')
print(df.dtypes)

In [ ]:
# Resumen estadístico
df.describe()

### 1.2 Manejar datos missing

In [ ]:
# Comprobamos si hay valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum())
print()
print('Total de nulos:', df.isnull().sum().sum())

No hay datos missing, así que no necesitamos hacer imputación ni eliminar filas.

### 1.3 Manejar datos categóricos

La columna `Genero` es categórica (Male/Female). Vamos a codificarla con Label Encoding para poder usarla en los algoritmos.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Codificación Label Encoding para el género
le = LabelEncoder()
df['Genero_cod'] = le.fit_transform(df['Genero'])
print('Mapeo del Label Encoder:')
print(dict(zip(le.classes_, le.transform(le.classes_))))
print()
df[['Genero', 'Genero_cod']].head()

In [ ]:
# También vamos a crear variables dummy para mostrar que sabemos hacerlo
# Aunque para clustering usaremos el Label Encoding
df_dummies = pd.get_dummies(df['Genero'], prefix='Genero', drop_first=True)
print('Variables dummy (con drop_first=True para evitar la trampa de las variables dummy):')
df_dummies.head()

Usamos `drop_first=True` para evitar la trampa de las variables dummy. Si tenemos 2 categorías (Male, Female), solo necesitamos 1 columna dummy porque la otra se deduce.

### 1.4 Preparar los datos para clustering

Eliminamos el CustomerID porque es solo un identificador y no aporta info útil para agrupar. También quitamos la columna original de Genero (nos quedamos con la codificada).

In [ ]:
# Seleccionamos las columnas que nos interesan para el clustering
# Quitamos CustomerID (no aporta) y Genero (usamos la codificada)
df_cluster = df[['Genero_cod', 'Edad', 'Ingresos Anuales (K)', 'Puntaje Gasto']].copy()
df_cluster.head()

### 1.5 Separar datos en conjuntos de entrenamiento y test

In [ ]:
from sklearn.model_selection import train_test_split

# Separamos en train y test (70% / 30%)
# En clustering no es tan habitual como en supervisado, pero lo hacemos porque lo pide el enunciado
X_train, X_test = train_test_split(df_cluster, test_size=0.3, random_state=42)
print('Tamaño entrenamiento:', X_train.shape)
print('Tamaño test:', X_test.shape)

### 1.6 Estandarizar los datos

Estandarizamos para que todas las variables tengan la misma escala. Esto es importante en clustering porque los algoritmos se basan en distancias, y si una variable tiene valores mucho más grandes que otra, pesará más.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Lo convertimos a DataFrame para que sea más fácil de manejar
X_train_scaled = pd.DataFrame(X_train_scaled, columns=df_cluster.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=df_cluster.columns)

print('Datos estandarizados (primeras filas):')
X_train_scaled.head()

Para el clustering vamos a usar el dataset completo estandarizado (no separado en train/test), ya que en aprendizaje no supervisado lo normal es usar todos los datos. También vamos a trabajar con un subconjunto de 2 variables (Ingresos y Puntaje Gasto) para poder visualizar mejor los clusters en 2D.

In [ ]:
# Dataset completo estandarizado para clustering
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(df_cluster)
X_full_scaled = pd.DataFrame(X_full_scaled, columns=df_cluster.columns)

# Subconjunto con solo Ingresos y Puntaje Gasto (para visualización 2D)
X_2d = df[['Ingresos Anuales (K)', 'Puntaje Gasto']].values
scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

print('Shape datos completos escalados:', X_full_scaled.shape)
print('Shape datos 2D escalados:', X_2d_scaled.shape)

### 1.7 EDA básico: Análisis de correlaciones, pairplot, mapas de calor

In [ ]:
# Distribución del género
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df['Genero'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'])
axes[0].set_title('Distribución del Género')
axes[0].set_ylabel('Cantidad')

# Distribución de la edad
axes[1].hist(df['Edad'], bins=15, color='steelblue', edgecolor='black')
axes[1].set_title('Distribución de la Edad')
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de Ingresos y Puntaje de Gasto
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df['Ingresos Anuales (K)'], bins=15, color='mediumseagreen', edgecolor='black')
axes[0].set_title('Distribución de Ingresos Anuales')
axes[0].set_xlabel('Ingresos (K)')

axes[1].hist(df['Puntaje Gasto'], bins=15, color='coral', edgecolor='black')
axes[1].set_title('Distribución del Puntaje de Gasto')
axes[1].set_xlabel('Puntaje')

plt.tight_layout()
plt.show()

In [ ]:
# Pairplot para ver relaciones entre las variables
sns.pairplot(df[['Edad', 'Ingresos Anuales (K)', 'Puntaje Gasto', 'Genero']], hue='Genero')
plt.suptitle('Pairplot de las variables', y=1.02)
plt.show()

In [ ]:
# Mapa de calor con las correlaciones
plt.figure(figsize=(8, 6))
corr = df_cluster.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Mapa de calor - Correlaciones')
plt.show()

In [ ]:
# Scatter plot de Ingresos vs Puntaje de Gasto
plt.figure(figsize=(8, 6))
plt.scatter(df['Ingresos Anuales (K)'], df['Puntaje Gasto'], c='steelblue', s=50, alpha=0.6)
plt.xlabel('Ingresos Anuales (K)')
plt.ylabel('Puntaje de Gasto')
plt.title('Ingresos vs Puntaje de Gasto')
plt.show()

### 1.8 Conclusiones del EDA

- El dataset tiene 200 clientes con 5 variables. No hay datos nulos.
- Hay un poco más de mujeres (112) que de hombres (88).
- Las edades van de 18 a 70 años, con la mayoría entre 30 y 40.
- Los ingresos anuales van de 15K a 137K.
- El puntaje de gasto va de 1 a 99.
- Las correlaciones entre variables son bastante bajas, lo cual es bueno porque no hay redundancia.
- En el scatter de Ingresos vs Puntaje de Gasto se intuyen ya algunos grupos de clientes, lo que sugiere que el clustering va a funcionar bien.
- No eliminamos ninguna columna porque las correlaciones son bajas y todas aportan información.

---
## 2. CLUSTERING (5 puntos)

### 2.1 Algoritmo K-Means (1 punto)

K-Means busca los centroides de los grupos. Funciona asignando cada punto al centroide más cercano y recalculando los centroides iterativamente hasta que converge.

Primero usamos el método del codo para encontrar el número óptimo de clusters.

In [ ]:
from sklearn.cluster import KMeans

# Método del codo
# WCSS = Within-Cluster Sum of Squares (suma de los cuadrados dentro del cluster)
# Lo que hace es medir lo compactos que son los clusters
# Cuanto menor es el WCSS, más compactos están los datos dentro de cada grupo

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', n_init=10, max_iter=300, random_state=0)
    kmeans.fit(X_2d_scaled)
    wcss.append(kmeans.inertia_)  # inertia_ es el WCSS

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, 'bo-')
plt.title('Método del Codo')
plt.xlabel('Número de Clusters')
plt.ylabel('WCSS')
plt.grid(True, alpha=0.3)
plt.show()

El codo se ve claramente en k=5. A partir de ahí la reducción de WCSS es mucho menor, así que elegimos 5 clusters.

In [ ]:
# Aplicamos K-Means con 5 clusters
kmeans = KMeans(n_clusters=5, init='k-means++', n_init=10, max_iter=300, random_state=0)
y_kmeans = kmeans.fit_predict(X_2d_scaled)

print('Etiquetas asignadas a cada cliente:')
print(y_kmeans)
print()
print('Número de clusters:', len(np.unique(y_kmeans)))

In [ ]:
# Visualización de los clusters
plt.figure(figsize=(10, 8))

colores = ['green', 'blue', 'magenta', 'cyan', 'red']
nombres = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3', 'Cluster 4']

for i in range(5):
    plt.scatter(X_2d_scaled[y_kmeans == i, 0], X_2d_scaled[y_kmeans == i, 1],
                s=60, c=colores[i], label=nombres[i])

# Centroides
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            s=200, marker='X', c='black', label='Centroides')

plt.title('Clusters K-Means (Ingresos vs Puntaje Gasto)')
plt.xlabel('Ingresos Anuales (estandarizado)')
plt.ylabel('Puntaje de Gasto (estandarizado)')
plt.legend()
plt.show()

In [ ]:
# Conteo del número de clientes por cluster
for i in range(5):
    print(f'Cluster {i}: {len(X_2d_scaled[y_kmeans == i])} clientes')

K-Means ha separado bien a los clientes en 5 grupos. Podemos ver que hay:
- Clientes con ingresos altos y gasto alto
- Clientes con ingresos altos y gasto bajo
- Clientes con ingresos bajos y gasto alto
- Clientes con ingresos bajos y gasto bajo
- Clientes con ingresos y gasto medio

### 2.2 Algoritmo DBSCAN (1 punto)

DBSCAN es un algoritmo basado en densidad. A diferencia de K-Means, no necesitamos decirle cuántos clusters queremos. Agrupa los puntos que están en zonas densas y marca como ruido los puntos aislados.

Tiene dos parámetros principales:
- **eps**: el radio de la vecindad. Define a qué distancia máxima se consideran vecinos dos puntos.
- **min_samples**: el mínimo de puntos que tiene que haber en esa vecindad para que se forme un cluster.

In [ ]:
from sklearn.cluster import DBSCAN

# Aplicamos DBSCAN
# Probamos con eps=0.5 y min_samples=5 (valores habituales para empezar)
dbscan = DBSCAN(eps=0.5, min_samples=5)
y_dbscan = dbscan.fit_predict(X_2d_scaled)

print('Etiquetas DBSCAN:', np.unique(y_dbscan))
print('Número de clusters encontrados:', len(set(y_dbscan)) - (1 if -1 in y_dbscan else 0))
print('Puntos de ruido:', (y_dbscan == -1).sum())

In [ ]:
# Visualización de los clusters DBSCAN
plt.figure(figsize=(10, 8))

# Pintamos cada cluster de un color diferente
clusters_unicos = set(y_dbscan)
colores_dbscan = plt.cm.Spectral(np.linspace(0, 1, len(clusters_unicos)))

for k, col in zip(sorted(clusters_unicos), colores_dbscan):
    if k == -1:
        # Los puntos de ruido los pintamos en negro
        col = 'black'
        label = 'Ruido'
    else:
        label = f'Cluster {k}'
    
    mask = (y_dbscan == k)
    plt.scatter(X_2d_scaled[mask, 0], X_2d_scaled[mask, 1],
                s=60, c=[col], label=label)

plt.title('Clusters DBSCAN (Ingresos vs Puntaje Gasto)')
plt.xlabel('Ingresos Anuales (estandarizado)')
plt.ylabel('Puntaje de Gasto (estandarizado)')
plt.legend()
plt.show()

In [ ]:
# Conteo de elementos por cluster
from collections import Counter
conteo = Counter(y_dbscan)
for cluster, cantidad in sorted(conteo.items()):
    if cluster == -1:
        print(f'Ruido: {cantidad} clientes')
    else:
        print(f'Cluster {cluster}: {cantidad} clientes')

DBSCAN identifica los clusters de forma diferente a K-Means. Es sensible a los parámetros eps y min_samples. Además puede detectar puntos de ruido, que son clientes que no encajan bien en ningún grupo.

### 2.3 Algoritmo Propagación de Afinidad (1 punto)

Propagación de Afinidad es un algoritmo que no necesita que le digamos cuántos clusters queremos. Funciona intercambiando mensajes entre los datos (responsabilidad y disponibilidad) hasta que converge y decide quiénes son los ejemplares (centros) de cada grupo.

Tal como vimos en clase, calcula una matriz de similitud y luego itera con las matrices de responsabilidad y disponibilidad hasta encontrar los ejemplares óptimos.

In [ ]:
from sklearn.cluster import AffinityPropagation

# Aplicamos Propagación de Afinidad
af = AffinityPropagation(random_state=0)
y_af = af.fit_predict(X_2d_scaled)

print('Etiquetas:', np.unique(y_af))
print('Número de clusters encontrados:', len(np.unique(y_af)))
print('Índices de los ejemplares (centroides):', af.cluster_centers_indices_)

In [ ]:
# Visualización de los clusters de Propagación de Afinidad
plt.figure(figsize=(10, 8))

labels = af.labels_
cluster_centers = af.cluster_centers_

colores_af = plt.cm.Spectral(np.linspace(0, 1, len(np.unique(labels))))

for k, col in zip(np.unique(labels), colores_af):
    mask = (labels == k)
    plt.scatter(X_2d_scaled[mask, 0], X_2d_scaled[mask, 1],
                c=[col], s=60, label=f'Cluster {k}')
    # Marcamos el ejemplar (centro) del cluster
    plt.scatter(cluster_centers[k, 0], cluster_centers[k, 1],
                marker='X', c=[col], s=200, edgecolor='k')

plt.title('Clusters Propagación de Afinidad (Ingresos vs Puntaje Gasto)')
plt.xlabel('Ingresos Anuales (estandarizado)')
plt.ylabel('Puntaje de Gasto (estandarizado)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Conteo de elementos por cluster
conteo_af = Counter(y_af)
for cluster, cantidad in sorted(conteo_af.items()):
    print(f'Cluster {cluster}: {cantidad} clientes')

Propagación de Afinidad ha encontrado sus propios clusters sin que le digamos cuántos. Tiende a crear más clusters que K-Means porque es más granular. Cada cluster tiene un ejemplar que es un dato real del dataset (no un punto calculado como los centroides de K-Means).

### 2.4 Métricas - Comparativa de modelos (2 puntos)

Vamos a comparar los tres algoritmos usando dos métricas:
- **Coeficiente de Silueta**: mide lo bien que cada punto encaja en su cluster. Va de -1 a 1. Cuanto más alto, mejor.
- **Índice de Davies-Bouldin**: mide la similitud entre clusters. Cuanto más bajo, mejor (clusters más separados y compactos).

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Calculamos las métricas para cada algoritmo
# Solo podemos calcular si hay al menos 2 clusters y no hay un solo cluster

# K-Means
sil_km = silhouette_score(X_2d_scaled, y_kmeans)
db_km = davies_bouldin_score(X_2d_scaled, y_kmeans)

# DBSCAN - quitamos los puntos de ruido para calcular las métricas
mask_no_ruido = y_dbscan != -1
if len(set(y_dbscan[mask_no_ruido])) >= 2:
    sil_db = silhouette_score(X_2d_scaled[mask_no_ruido], y_dbscan[mask_no_ruido])
    db_db = davies_bouldin_score(X_2d_scaled[mask_no_ruido], y_dbscan[mask_no_ruido])
else:
    sil_db = float('nan')
    db_db = float('nan')

# Propagación de Afinidad
sil_af = silhouette_score(X_2d_scaled, y_af)
db_af = davies_bouldin_score(X_2d_scaled, y_af)

print('=== COMPARATIVA DE MÉTRICAS ===')
print(f'{"Algoritmo":<30} {"Silueta":<15} {"Davies-Bouldin":<15}')
print('-' * 60)
print(f'{"K-Means (k=5)":<30} {sil_km:<15.4f} {db_km:<15.4f}')
print(f'{"DBSCAN":<30} {sil_db:<15.4f} {db_db:<15.4f}')
print(f'{"Propagación de Afinidad":<30} {sil_af:<15.4f} {db_af:<15.4f}')

In [ ]:
# Visualización comparativa lado a lado
# Similar a como lo hace el profesor en el notebook de métricas
fig = plt.figure(figsize=(18, 5))
plt.set_cmap('gist_ncar')

# K-Means
ax1 = fig.add_subplot(1, 3, 1)
ax1.scatter(X_2d_scaled[:, 0], X_2d_scaled[:, 1],
            c=y_kmeans, s=100, linewidth=0.5, edgecolors='black')
ax1.set_title(f'K-MEANS\n\nSilueta = {sil_km:.2f} | Davies-Bouldin = {db_km:.2f}', fontsize=10)
ax1.axis('off')

# DBSCAN
ax2 = fig.add_subplot(1, 3, 2)
ax2.scatter(X_2d_scaled[:, 0], X_2d_scaled[:, 1],
            c=y_dbscan, s=100, linewidth=0.5, edgecolors='black')
ax2.set_title(f'DBSCAN\n\nSilueta = {sil_db:.2f} | Davies-Bouldin = {db_db:.2f}', fontsize=10)
ax2.axis('off')

# Propagación de Afinidad
ax3 = fig.add_subplot(1, 3, 3)
ax3.scatter(X_2d_scaled[:, 0], X_2d_scaled[:, 1],
            c=y_af, s=100, linewidth=0.5, edgecolors='black', alpha=0.85)
ax3.set_title(f'PROPAGACIÓN DE AFINIDAD\n\nSilueta = {sil_af:.2f} | Davies-Bouldin = {db_af:.2f}', fontsize=10)
ax3.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# También vemos cómo varían las métricas de K-Means para distintos valores de K
# Igual que en el notebook del profe
fig = plt.figure(figsize=(24, 10))

for k in range(2, 10):
    clusters_temp = KMeans(n_clusters=k, random_state=0)
    clusters_temp.fit_predict(X_2d_scaled)
    
    silueta = silhouette_score(X_2d_scaled, clusters_temp.labels_)
    davies = davies_bouldin_score(X_2d_scaled, clusters_temp.labels_)
    
    ax = fig.add_subplot(2, 4, k-1)
    ax.scatter(X_2d_scaled[:, 0], X_2d_scaled[:, 1],
               c=clusters_temp.labels_, s=100,
               linewidth=0.5, edgecolors='black', alpha=0.85)
    ax.set_title(f'K={k}\n\nSilueta = {silueta:.2f}\nDavies-Bouldin = {davies:.2f}', fontsize=12)
    ax.axis('off')

plt.suptitle('K-Means para diferentes valores de K', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Conclusiones de la comparativa

- **K-Means** con k=5 da buenos resultados. El coeficiente de silueta confirma que los clusters están bien separados. Es el algoritmo que mejor se adapta a estos datos porque los grupos tienen forma más o menos esférica.
- **DBSCAN** tiene más dificultad con este dataset porque los clusters no están claramente separados por densidad. Algunos puntos quedan como ruido.
- **Propagación de Afinidad** tiende a crear más clusters de los necesarios, aunque las métricas pueden ser razonables.
- Mirando las métricas juntas (silueta alta y Davies-Bouldin bajo), K-Means con k=5 es la mejor opción para este dataset.
- El gráfico con distintos valores de K confirma que k=5 es donde se obtienen las mejores métricas.

---
## 3. MARKET BASKET - APRIORI (2 puntos)

El algoritmo Apriori sirve para encontrar conjuntos de ítems frecuentes en datos transaccionales. Se usa mucho en análisis de la cesta de la compra: si un cliente compra X e Y juntos, es probable que también compre Z.

Como vimos en clase, el concepto clave es el **soporte**: la frecuencia relativa con la que aparece un conjunto de ítems en las transacciones.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

### 3.1 Importar / crear dataset

Creamos un dataset de transacciones de un supermercado simulado. Cada lista es lo que compró un cliente.

In [ ]:
# Dataset de transacciones de un supermercado
transacciones = [
    ['Pan', 'Leche', 'Huevos', 'Mantequilla'],
    ['Pan', 'Leche', 'Huevos'],
    ['Pan', 'Leche', 'Cerveza', 'Pañales'],
    ['Pan', 'Cerveza', 'Pañales'],
    ['Leche', 'Huevos', 'Mantequilla'],
    ['Pan', 'Leche', 'Huevos', 'Mantequilla', 'Cerveza'],
    ['Pan', 'Leche'],
    ['Huevos', 'Mantequilla'],
    ['Pan', 'Leche', 'Cerveza'],
    ['Pan', 'Huevos', 'Mantequilla'],
    ['Leche', 'Huevos', 'Cerveza', 'Pañales'],
    ['Pan', 'Leche', 'Huevos'],
    ['Pan', 'Leche', 'Pañales'],
    ['Cerveza', 'Pañales'],
    ['Pan', 'Leche', 'Huevos', 'Cerveza', 'Pañales']
]

print(f'Número de transacciones: {len(transacciones)}')

### 3.2 Preprocesamiento - TransactionEncoder (OneHotEncoding)

In [ ]:
# Transformamos las transacciones a formato OneHotEncoding con TransactionEncoder
te = TransactionEncoder()
te_array = te.fit(transacciones).transform(transacciones)
df_trans = pd.DataFrame(te_array, columns=te.columns_)
df_trans

Cada fila es una transacción y cada columna un producto. True significa que el cliente compró ese producto.

### 3.3 Aplicar algoritmo Apriori

In [ ]:
# Aplicamos Apriori con un soporte mínimo del 30%
# use_colnames=True para ver los nombres de los productos en vez de números
itemsets_frecuentes = apriori(df_trans, min_support=0.3, use_colnames=True)
itemsets_frecuentes['longitud'] = itemsets_frecuentes['itemsets'].apply(lambda x: len(x))
itemsets_frecuentes

La columna `support` indica la frecuencia relativa de cada conjunto de ítems. Por ejemplo, un soporte de 0.60 significa que aparece en el 60% de las transacciones.

### 3.4 Filtrado sobre el itemset y el support

In [ ]:
# Filtrar conjuntos de 2 ítems con soporte >= 40%
filtro = itemsets_frecuentes[(itemsets_frecuentes['longitud'] == 2) & 
                             (itemsets_frecuentes['support'] >= 0.4)]
print('Pares de productos frecuentes (soporte >= 40%):')
filtro

In [ ]:
# Filtrar por un producto concreto
# Por ejemplo, buscar todos los itemsets que contengan 'Pan'
filtro_pan = itemsets_frecuentes[itemsets_frecuentes['itemsets'].apply(lambda x: 'Pan' in x)]
print('Itemsets que contienen Pan:')
filtro_pan

In [ ]:
# Buscar un itemset concreto usando frozenset
# Como vimos en clase, el orden no importa en los frozensets
print('Buscando el par {Pan, Leche}:')
itemsets_frecuentes[itemsets_frecuentes['itemsets'] == frozenset({'Pan', 'Leche'})]

In [ ]:
# Generamos reglas de asociación a partir de los itemsets frecuentes
reglas = association_rules(itemsets_frecuentes, metric='confidence', min_threshold=0.5)
reglas = reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
reglas.sort_values('lift', ascending=False)

### 3.5 Conclusiones del Market Basket

- Los productos que más se compran juntos son **Pan y Leche**, con un soporte alto, lo que indica que aparecen juntos en muchas transacciones.
- También hay asociaciones frecuentes entre **Cerveza y Pañales**, que es un ejemplo clásico de market basket analysis.
- El algoritmo Apriori nos permite descubrir estas asociaciones de forma automática, lo que es muy útil para:
  - Colocar productos relacionados cerca en el supermercado
  - Hacer ofertas cruzadas (si compras X, te descuento Y)
  - Entender los patrones de compra de los clientes
- El **lift** mayor que 1 indica que la asociación es positiva: comprar un producto aumenta la probabilidad de comprar el otro.

---
## 4. DETECCIÓN DE ANOMALÍAS - BOSQUES DE AISLAMIENTO (2 puntos)

Los Bosques de Aislamiento (Isolation Forests) son una técnica para detectar datos anómalos. La idea es que los datos raros son más fáciles de separar que los normales, así que quedan en las ramas cercanas a la raíz del árbol.

Como vimos en clase:
- Los datos extremos se encuentran en posiciones altas del árbol (cerca de la raíz)
- Los datos normales se agrupan en la parte inferior
- El nivel de contaminación define qué porcentaje de los datos consideramos anómalos

Vamos a usar el dataset de **carros usados** (km vs precio), igual que el profe en el notebook.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Cargamos el dataset de carros
carros = np.loadtxt('carros.csv', delimiter=',', skiprows=1)  # km / precio
print('Shape:', carros.shape)
print('Primeras filas:')
print(carros[:5])

In [ ]:
# Normalizamos los datos con MinMaxScaler para que estén en el rango [0, 1]
scaler_carros = MinMaxScaler()
carros_norm = scaler_carros.fit_transform(carros)

print('Datos normalizados (primeras filas):')
print(carros_norm[:5])

### 4.1 Entrenar el modelo con distintos niveles de contaminación

In [ ]:
# Probamos con 3 niveles de contaminación: 1%, 5% y 10%
niveles_cont = [0.01, 0.05, 0.1]
resultados = np.zeros((3, carros_norm.shape[0]))

for i, c in enumerate(niveles_cont):
    modelo = IsolationForest(contamination=c, random_state=42)
    modelo.fit(carros_norm)
    resultados[i] = modelo.predict(carros_norm)  # 1 = normal, -1 = anomalía

# Conteo de anomalías para cada nivel
for i, c in enumerate(niveles_cont):
    n_anomalias = (resultados[i] == -1).sum()
    print(f'Contaminación {c}: {n_anomalias} anomalías detectadas de {carros_norm.shape[0]} datos')

### 4.2 Visualización gráfica de los resultados

In [ ]:
# Visualización similar a la del profesor
plt.set_cmap('jet')
fig = plt.figure(figsize=(16, 5))

for i in range(len(niveles_cont)):
    ax = fig.add_subplot(1, 3, i+1)
    
    # Datos anómalos marcados con cuadrados azules
    ax.scatter(carros_norm[resultados[i] == -1][:, 0],
               carros_norm[resultados[i] == -1][:, 1],
               c='skyblue', marker='s', s=300, label='Anómalo')
    
    # Todos los datos
    ax.scatter(carros_norm[:, 0],
               carros_norm[:, 1],
               c=range(carros_norm.shape[0]), marker='x',
               s=300, alpha=0.6, label='Normal')
    
    ax.set_title(f'Contaminación: {niveles_cont[i]:.2f}', size=14, color='purple')
    ax.set_xlabel('Kms recorridos (normalizado)', size=10)
    ax.set_ylabel('Precio (normalizado)', size=10)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Vemos cuáles son los datos detectados como anómalos con contaminación 0.05
print('Datos anómalos (contaminación 5%):')
anomalos = carros[resultados[1] == -1]
print('Kms\t\tPrecio')
for fila in anomalos:
    print(f'{fila[0]:.0f}\t\t{fila[1]:.0f}')

In [ ]:
# También probamos con los datos sin normalizar para ver la diferencia
fig = plt.figure(figsize=(16, 5))
resultados_sin_norm = np.zeros((3, carros.shape[0]))

for i, c in enumerate(niveles_cont):
    modelo_sn = IsolationForest(contamination=c, random_state=42)
    modelo_sn.fit(carros)
    resultados_sin_norm[i] = modelo_sn.predict(carros)

for i in range(len(niveles_cont)):
    ax = fig.add_subplot(1, 3, i+1)
    ax.scatter(carros[resultados_sin_norm[i] == -1][:, 0],
               carros[resultados_sin_norm[i] == -1][:, 1],
               c='skyblue', marker='s', s=300)
    ax.scatter(carros[:, 0], carros[:, 1],
               c=range(carros.shape[0]), marker='x', s=300, alpha=0.6)
    ax.set_title(f'Sin normalizar - Cont: {niveles_cont[i]:.2f}', size=14, color='purple')
    ax.set_xlabel('Kms recorridos', size=10)
    ax.set_ylabel('Precio ($)', size=10)

plt.tight_layout()
plt.show()

### 4.3 Conclusiones de la detección de anomalías

- Con contaminación al 1%, solo se detectan los puntos más extremos.
- Con el 5%, se detectan más datos anómalos, que son coches que tienen un precio muy alto con muchos kilómetros o un precio muy bajo con pocos kilómetros. Ambos casos son sospechosos.
- Con el 10%, se amplía la detección a más puntos que se alejan de la tendencia general.
- Los datos normalizados y sin normalizar dan resultados similares, aunque normalizar es una buena práctica para que ambas variables tengan el mismo peso.
- En un caso real, estos datos anómalos podrían ser errores en los datos, fraudes, o simplemente coches especiales (de colección, con daños, etc.).
- El nivel de contaminación hay que ajustarlo según el dominio: en fraude bancario puede ser muy bajo (1% o menos) y en control de calidad puede ser más alto.